# PolliVision — fine-tuning a rover detector on free GPU

The PolliVision stack runs **without any training at all**: its default detector is
open-vocabulary and is prompted in natural language. This notebook is for the step
*after* that — distilling the slow open-vocabulary teacher into a small supervised
student that runs several times faster on rover compute and is tuned to your crop.

You do not need a GPU of your own. This runs on Colab's free tier in roughly
20–40 minutes for a few thousand images.

**Before starting**, produce a labelled dataset on your own machine (CPU is fine,
it is a batch job):

```bash
python tools/autolabel.py field_images/ --output dataset \
    --species pumpkin --teacher yoloe-11l-seg --preview
```

Then **review the labels in `dataset/preview/`**. This is not optional. The teacher
makes systematic mistakes — most often on partly occluded flowers and on sex, where
it has no depth to work with — and a student trained on unreviewed labels reproduces
those mistakes with more confidence and no way to tell it is wrong.

Upload the reviewed `dataset/` folder to Google Drive, then run the cells below.


## 1. Check the GPU and install


In [ ]:
!nvidia-smi
# Free-tier Colab usually gives a T4. Anything here is far faster than CPU training.


In [ ]:
!pip install -q ultralytics
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())


## 2. Mount the reviewed dataset


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Point this at wherever you uploaded the reviewed dataset.
DATASET = '/content/drive/MyDrive/pollivision/dataset'

import yaml, pathlib
cfg = yaml.safe_load(open(f'{DATASET}/data.yaml'))
# autolabel.py writes an absolute path from the machine that produced the
# dataset; rewrite it for this runtime.
cfg['path'] = DATASET
yaml.safe_dump(cfg, open(f'{DATASET}/data.yaml','w'), sort_keys=False)
print(cfg)


In [ ]:
# Sanity-check class balance before spending GPU time on it.
from collections import Counter
counts = Counter()
for split in ('train','val'):
    for p in pathlib.Path(f'{DATASET}/labels/{split}').glob('*.txt'):
        for line in p.read_text().split(chr(10)):
            if line.strip():
                counts[cfg['names'][int(line.split()[0])]] += 1
for name in cfg['names']:
    print(f'{name:<18} {counts.get(name,0)}')

female = counts.get('flower_female', 0)
male = counts.get('flower_male', 0)
if female and male and max(male,female)/max(min(male,female),1) > 4:
    print(chr(10) + 'Warning: the sexes are badly imbalanced. Cucurbits genuinely produce',
          'more male flowers than female, so some skew is real — but beyond about 4:1',
          'the student will learn to just guess male. Collect more pistillate examples',
          'or set class weights.')


## 3. Train

`yolo11s` is the recommended student: it is small enough to export and run on a
rover SBC, and large enough to benefit from a few thousand labelled images. Drop to
`yolo11n` if inference latency matters more than the last few points of accuracy.

The augmentation settings below are deliberately tuned for this problem:

- **`hsv_h` is kept low.** Flower sex and pollen load are partly *chroma* cues, and
  aggressive hue jitter teaches the model to ignore exactly the signal it needs.
- **`hsv_v` and `hsv_s` are generous.** Outdoor illumination genuinely varies by
  orders of magnitude between overcast and direct sun.
- **`flipud` stays at 0.** A pistillate flower is defined by having an ovary
  *below* it. Vertical flips destroy that relationship and teach the model that up
  and down are interchangeable, which is the one thing it must not believe.
- **`degrees` is small** for the same reason: large rotations blur the same cue.


In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11s.pt')
results = model.train(
    data=f'{DATASET}/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    patience=25,            # stop early rather than overfit a small dataset
    optimizer='auto',
    cos_lr=True,
    hsv_h=0.010,            # minimal hue jitter: sex and pollen cues are chroma
    hsv_s=0.70,
    hsv_v=0.45,
    degrees=8.0,            # small: the ovary-below-corolla relation must survive
    translate=0.12,
    scale=0.45,             # flowers appear at very different ranges
    fliplr=0.5,
    flipud=0.0,             # never: it inverts the defining female cue
    mosaic=1.0,
    close_mosaic=15,        # disable mosaic for the last epochs to settle boxes
    project='pollivision',
    name='student',
)


## 4. Evaluate

Look past the headline mAP. The number that matters operationally is the
**female-class recall**: a missed pistillate flower is a fruit that never sets,
while a missed staminate flower usually just means collecting from the next one.


In [ ]:
metrics = model.val()
print(f'mAP50    {metrics.box.map50:.4f}')
print(f'mAP50-95 {metrics.box.map:.4f}')
print()
for i, name in enumerate(cfg['names']):
    try:
        print(f'{name:<18} P {metrics.box.p[i]:.3f}  R {metrics.box.r[i]:.3f}  '
              f'mAP50 {metrics.box.ap50[i]:.3f}')
    except (IndexError, TypeError):
        print(f'{name:<18} (no validation instances)')


## 5. Export for the rover

Pick the format that matches your rover's compute. `ncnn` is usually the right
answer for an ARM SBC; `openvino` for an Intel one; `onnx` if you are unsure.


In [ ]:
model.export(format='onnx', imgsz=640)   # portable default
# model.export(format='ncnn', imgsz=640)      # ARM SBCs (Raspberry Pi etc.)
# model.export(format='openvino', imgsz=640)  # Intel SBCs


In [ ]:
import shutil
OUT = '/content/drive/MyDrive/pollivision/trained'
shutil.copytree('pollivision/student/weights', OUT, dirs_exist_ok=True)
print('Weights copied to', OUT)


## 6. Wire it into the rover

Download `best.pt` (or the exported model) and enable it as a fusion member.
Keep the open-vocabulary backend switched on: the student is more accurate on the
flowers it was trained for, and the teacher is the recall safety net for the ones
it was not.

```yaml
# my_rover.yaml
detector:
  backends:
    - name: finetuned
      weights: /path/to/best.pt
      enabled: true
      weight: 1.5          # trust the supervised student most
      conf: 0.25
    - name: yoloe
      weights: yoloe-11s-seg
      enabled: true
      weight: 1.0          # open-vocabulary recall safety net
      conf: 0.10
  fusion:
    require_votes: 1       # raise to 2 for precision over recall
```

```bash
pollivision run config --config my_rover --species pumpkin
```

A student whose class names contain `male` / `female` is detected automatically and
its prediction is fed to the sex head as an additional cue — it does not replace the
other cues, it joins them.
